# 💬 Ask the Imaging Data Commons

Ask questions in plain English about **free public cancer images** (CT, MRI, PET, pathology
slides…) from the NCI Imaging Data Commons. You don't need to know any programming.

Each question goes to the **IDC assistant (MCP `ask` tool)**. The answer comes back with:
- the **numbers** (patients, scans, size),
- **links that open the scans in your web browser**,
- the **licence**, which tells you what you are allowed to do with the data,
- and, for the curious, the **query it used** (hidden under *Show the query*).

### How to use it
1. Press **▶ Run** on the two *Setup* cells below (or *Run → Run All*).
2. Type your question in the **chat box** and press **Ask**, or edit and run any example.
3. Nothing is downloaded unless you ask for a download **and** run the download cell.

## Setup (run once)

In [1]:
# @title 1 · Install (about a minute the first time)
import sys
!{sys.executable} -m pip install -q idc-index ipywidgets

In [2]:
# @title 2 · Connect to the IDC assistant
import html
import logging
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

# The assistant lives in idc_explorer/assistant in this repository.
for folder in (Path("../../idc_explorer/assistant"), Path("idc_explorer/assistant"), Path("assistant")):
    if (folder / "tools.py").exists():
        sys.path.insert(0, str(folder.resolve()))
        break
else:
    raise FileNotFoundError("Open this notebook from the repository so it can find idc_explorer/assistant.")

import tools  # the MCP tools: tools.TOOLS and tools.call_tool(name, arguments)

def quiet() -> None:
    # Keep answers free of idc-index log lines (its download functions turn them back on).
    logging.getLogger("idc_index.index").setLevel(logging.WARNING)


tools._client()     # open the catalogue once, before silencing its log
quiet()
last_table = None   # the result of the last SQL playground query, as a table

HIDE = {"StudyInstanceUID", "SeriesInstanceUID", "series_aws_url"}
NICE = {"collection_id": "Dataset", "collection_name": "Name", "PatientID": "Patient",
        "Modality": "Type", "BodyPartExamined": "Body part", "SeriesDescription": "Description",
        "series_size_MB": "Size (MB)", "license_short_name": "Licence", "total_size_MB": "Size (MB)",
        "subjects": "Patients", "cancer_types": "Cancer types", "tumor_locations": "Locations"}
last = {}


def chat(question: str) -> None:
    """Send one question to the MCP `ask` tool and show the answer."""
    quiet()
    answer = tools.call_tool("ask", {"question": question, "confirm_large": False})
    last.clear()
    last.update(answer, wants_download=bool(re.search(r"\bdownload", question, re.I)))
    parts = [f"<p style='font-size:1.05em'>🧑 <b>{html.escape(question)}</b></p>"]
    if not answer["understood"]:
        parts.append(f"<p>🤖 {html.escape(answer['summary'])}</p>")
        parts += [f"<p>💡 {html.escape(n)}</p>" for n in answer["notes"]]
        display(HTML("".join(parts)))
        return
    parts.append(f"<p>🤖 {html.escape(answer['summary'])}</p>")
    display(HTML("".join(parts)))

    rows = pd.DataFrame(answer["rows"])
    if len(rows):
        display(rows[[c for c in rows.columns if c not in HIDE]].rename(columns=NICE))

    out = []
    cohort = answer.get("cohort")
    if cohort:
        out.append(f"<p>📦 <b>All matching scans:</b> {cohort['patients']:,} patients · "
                   f"{cohort['series']:,} scans · {cohort['total_size_MB'] / 1000:,.1f} GB</p>")
    guard = answer.get("guardrail") or {}
    if guard.get("message"):
        out.append(f"<p>⚖️ {html.escape(guard['message'])}</p>")
    links = answer.get("viewer_links") or []
    if links:
        out.append("<p>👁️ <b>Open in your browser</b> (nothing is downloaded):</p><ul>" + "".join(
            f"<li><a href='{l['url']}' target='_blank'>{html.escape(str(l['collection_id']))} · "
            f"patient {html.escape(str(l['PatientID']))}</a></li>" for l in links) + "</ul>")
    for note in answer.get("notes") or []:
        out.append(f"<p style='color:#777'>ℹ️ {html.escape(note)}</p>")
    if last["wants_download"] and answer.get("manifest"):
        out.append("<p>⬇️ <b>Ready to download the scans in the table above.</b> "
                   "Run the <i>Download</i> cell below to fetch them.</p>")
    sql = (answer.get("work") or {}).get("idc_index_sql")
    if sql:
        out.append(f"<details><summary>Show the query</summary><pre>{html.escape(sql)}</pre></details>")
    display(HTML("".join(out)))


def download_last(folder: str = "downloads", max_mb: float = 500) -> None:
    """Download the scans listed in the last answer, only if that question asked to download."""
    if not last.get("wants_download"):
        print("Ask a question with the word 'download' first, for example: "
              "Download the smallest brain MRI series")
        return
    rows = [r for r in last.get("rows", []) if r.get("SeriesInstanceUID")]
    size = sum(r.get("series_size_MB") or 0 for r in rows)
    if not rows:
        print("That answer has no scans to download.")
        return
    if size > max_mb:
        print(f"{size:,.0f} MB is more than max_mb={max_mb}. Raise max_mb on purpose if you mean it.")
        return
    print("Licences:", ", ".join(sorted({str(r.get('license_short_name')) for r in rows})),
          "- please cite the data as each licence requires.")
    target = Path(folder)
    tools._client().download_from_selection(
        seriesInstanceUID=[r["SeriesInstanceUID"] for r in rows], downloadDir=str(target),
        dirTemplate="%collection_id/%PatientID/%Modality_%SeriesInstanceUID",
        use_s5cmd_sync=True, show_progress_bar=False)
    quiet()
    files = list(target.rglob("*.dcm"))
    print(f"✅ {len(files)} DICOM files ({sum(f.stat().st_size for f in files) / 1e6:,.1f} MB) "
          f"saved in {target.resolve()}")


def run_sql(sql: str, max_rows: int = 1000) -> None:
    """SQL playground: run one read-only query through the MCP `query_index` tool.

    The rows are shown and kept in `last_table` (a pandas table) for further use.
    """
    global last_table
    quiet()
    try:
        result = tools.call_tool("query_index", {"sql": sql, "max_rows": max_rows})
    except Exception as exc:
        display(HTML(f"<p style='color:#b00'>⚠️ The query did not run: {html.escape(str(exc))}</p>"))
        return
    frame = last_table = pd.DataFrame(result["rows"])
    note = f" (first {max_rows:,} only)" if result["truncated"] else ""
    display(HTML(f"<p>🧪 <b>{len(frame):,} rows</b>{note}</p>"))
    display(frame)
    python = (result.get("work") or {}).get("python")
    if python:
        display(HTML("<details><summary>Run the same query in your own Python</summary>"
                     f"<pre>{html.escape(python)}</pre></details>"))


print("Connected. Tools available:", ", ".join(t["name"] for t in tools.TOOLS))

Connected. Tools available: ask, query_index


## 💬 Chat box

Type a question and press **Ask** (or Enter). While you type, matching questions from the
question library are suggested. Each answer appears under the box.

In [3]:
import ipywidgets as widgets

import json

# Suggestions while you type: the 64 checked questions of the question library.
library = [json.loads(line)["question"]
           for line in (Path(tools.__file__).parent / "questions.jsonl").read_text().splitlines() if line.strip()]
box = widgets.Combobox(options=library, ensure_option=False,
                       placeholder="Type a question – matching suggestions appear as you type",
                       layout=widgets.Layout(width="70%"))
send = widgets.Button(description="Ask", button_style="primary")
answers = widgets.Output()


def on_ask(_):
    question = box.value.strip()
    if not question:
        return
    with answers:
        chat(question)
    box.value = ""


send.on_click(on_ask)
box.on_submit(on_ask)
display(widgets.HBox([box, send]), answers)

/var/folders/zp/kn2c7zks0q55p6_3s3c4b97h0000gn/T/ipykernel_83973/1013966537.py:25: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  box.on_submit(on_ask)


Output()

## 10 example questions

Run any of these, or change the words and run it again. The answer shown under each one
was produced when this notebook was generated.

### Number of patients

**1 · Patients in one dataset.** Dataset names are written in lowercase with underscores.

In [4]:
chat("How many patients are in rider_pilot?")

,patients,studies,series,Size (MB)
0,8,54,194,11224.7


**2 · Patients with a type of scan and a body part.**

In [5]:
chat("How many patients have brain MRI?")

,patients,studies,series,Size (MB)
0,83,171,266,7344.6


**3 · Add the patient's sex.**

In [6]:
chat("How many female patients have breast MRI?")

,patients,studies,series,Size (MB)
0,3265,5003,44005,2165730.5


### Explore the datasets

**4 · Which datasets cover a cancer.** Works for lung, breast, prostate, colon, liver, kidney and brain.

In [7]:
chat("Which collections have lung cancer?")

,Dataset,Name,Patients,Cancer types,Locations
0,nlst,NLST,26410,"Lung Cancer, Non-Cancer",Chest
1,eay131,EAY131,2813,"Adenocarcinoma, Adrenocortical Carcinoma, Anal...",Various
2,covid_19_ny_sbu,COVID-19-NY-SBU,1384,COVID-19 (non-cancer),Lung
3,lidc_idri,LIDC-IDRI,1010,"Lung Cancer, Non-Cancer, Metastatic disease",Chest
4,pdxnet,PDXNet,919,"Acinar Cell Carcinoma, Adrenal Cortical Carcin...","adrenal gland, bladder organ, bone element, br..."
5,fdg_pet_ct_lesions,FDG-PET-CT-Lesions,900,"Lymphoma, Melanoma, Non-small Cell Lung Cancer","lung, lymph node, skin of body"
6,tcga_luad,TCGA-LUAD,560,Lung Adenocarcinoma,Chest
7,mediastinal_lymph_node_seg,Mediastinal-Lymph-Node-SEG,513,"Breast Cancer, Non-small Cell Lung Cancer, Hod...",Lymph Node
8,tcga_lusc,TCGA-LUSC,504,Lung Squamous Cell Carcinoma,Lung
9,nsclc_radiomics,NSCLC-Radiomics,422,Lung Cancer,Lung


**5 · Licence: what am I allowed to do with it?** *CC BY* = any use with credit, *CC BY-NC* = no commercial use.

In [8]:
chat("What licence applies to chest CT?")

,Licence,series,patients,Size (MB)
0,CC BY 4.0,202655,27649,11651280.2
1,CC BY 3.0,3102,1608,288105.3
2,CC BY-NC 4.0,94,83,7833.8


### Look at the raw images

**6 · A few scans, with links that open them in your browser.**

In [9]:
chat("Show me a few chest CT series")

,Dataset,Patient,Type,Body part,Description,Size (MB),Licence
0,cptac_lscc,C3N-03486,CT,CHEST,NaN,0.13,CC BY 4.0
1,tcga_luad,TCGA-17-Z043,CT,CHEST,Monitoring,0.13,CC BY 3.0
2,tcga_luad,TCGA-17-Z043,CT,CHEST,PreMonitoring,0.13,CC BY 3.0


**7 · Pathology slides (microscope images of tissue).**

In [10]:
chat("Show me a few pathology slides")

,Dataset,Patient,Type,Body part,Description,Size (MB),Licence
0,htan_wustl,HTA12_15,SM,None,IMC,0.46,CC BY 4.0
1,tcga_stad,TCGA-BR-4371,SM,None,FFPE HE TP DX1,0.49,CC BY 3.0
2,tcga_stad,TCGA-BR-4294,SM,None,FFPE HE TP DX1,0.49,CC BY 3.0


### Analysis data
*Analysis results* are extra data made from the original images, such as outlines of organs
or tumours drawn by experts or AI. Normally they are left out. Say **analysis results** to include them.

**8 · Examples of segmentations (outlines), including analysis results.**

In [11]:
chat("Show me a few segmentations, including analysis results")

,Dataset,Patient,Type,Body part,Description,Size (MB),Licence
0,rider_pilot,1.3.6.1.4.1.9328.50.2.0072,SEG,None,None,0.0,CC BY 4.0
1,rider_pilot,1.3.6.1.4.1.9328.50.2.0072,SEG,None,None,0.0,CC BY 4.0
2,rider_pilot,1.3.6.1.4.1.9328.50.2.0072,SEG,None,None,0.0,CC BY 4.0


**9 · How many segmentations exist, including analysis results.**

In [12]:
chat("How many segmentation series, including analysis results?")

,series,patients,studies,Size (MB)
0,193295,43824,97295,19495350.1


### Download
**10 · Ask for a download.** The answer lists the scans and their size. Nothing is fetched yet.

In [13]:
chat("Download the smallest brain MRI series")

,Dataset,Patient,Type,Body part,Description,Size (MB),Licence
0,eay131,EAY131-854653,MR,BRAIN,AX_DIFF_ADC,0.08,CC BY 4.0
1,cmb_brca,MSB-01000,MR,BRAIN,AAHead_Scout_MPR_cor,0.17,CC BY 4.0
2,cmb_brca,MSB-01000,MR,BRAIN,AAHead_Scout_MPR_tra,0.17,CC BY 4.0


Now run this cell to really download those scans into a `downloads` folder next to this
notebook. It stops if the files are larger than `max_mb`, and the files are standard
**DICOM** images, which you can open with free viewers such as 3D Slicer, Horos, RadiAnt or Weasis.

In [14]:
download_last(folder="downloads", max_mb=500)

Licences: CC BY 4.0 - please cite the data as each licence requires.


✅ 7 DICOM files (0.4 MB) saved in /Users/taiaburrahman/Desktop/git/idc-index/solutions/03-notebook/downloads


---
## 🧪 SQL playground

For anyone who wants to ask the data **directly**. Write one SQL query, or start from an example.
It runs through the assistant's MCP **`query_index`** tool, which only allows reading:
one `SELECT` (or `WITH … SELECT`) query at a time, with up to 1,000 rows shown.

**Tables:** `index` (one row per scan) · `collections_index` (datasets) · `ct_index`,
`mr_index`, `pt_index` (scanner settings) · `seg_index` (outlines) · `analysis_results_index`
(derived data). Column descriptions: `tools._client().get_index_schema("index")`.

In [15]:
SQL_EXAMPLES = {
    "Patients and size per dataset": """SELECT collection_id, COUNT(DISTINCT PatientID) AS patients,
       COUNT(*) AS series, ROUND(SUM(series_size_MB) / 1000, 1) AS size_GB
FROM index GROUP BY collection_id ORDER BY patients DESC""",
    "Imaging types": """SELECT Modality, COUNT(*) AS series, COUNT(DISTINCT collection_id) AS datasets
FROM index GROUP BY Modality ORDER BY series DESC""",
    "Licences": """SELECT license_short_name, COUNT(*) AS series, ROUND(SUM(series_size_MB) / 1e6, 2) AS size_TB
FROM index GROUP BY 1 ORDER BY series DESC""",
    "Thin-slice CT per dataset": """SELECT i.collection_id, COUNT(*) AS series
FROM index i JOIN ct_index ct USING (SeriesInstanceUID)
WHERE ct.SliceThickness <= 1.5 GROUP BY 1 ORDER BY series DESC""",
    "What the segmentations outline": """SELECT outlined, AlgorithmType, COUNT(*) AS seg_series
FROM (SELECT UNNEST(SegmentedPropertyType_CodeMeanings) AS outlined, AlgorithmType FROM seg_index)
GROUP BY 1, 2 ORDER BY seg_series DESC LIMIT 20""",
    "Analysis results": """SELECT analysis_result_id, analysis_result_title, subjects, modalities, license_short_name
FROM analysis_results_index ORDER BY subjects DESC""",
}

pick = widgets.Dropdown(options=list(SQL_EXAMPLES), description="Example:",
                        layout=widgets.Layout(width="70%"))
editor = widgets.Textarea(value=SQL_EXAMPLES[pick.value], layout=widgets.Layout(width="100%", height="160px"))
run = widgets.Button(description="▶ Run query", button_style="primary")
results = widgets.Output()
pick.observe(lambda change: setattr(editor, "value", SQL_EXAMPLES[change["new"]]), names="value")


def on_run(_):
    results.clear_output()
    with results:
        run_sql(editor.value)


run.on_click(on_run)
display(pick, editor, run, results)

Dropdown(description='Example:', layout=Layout(width='70%'), options=('Patients and size per dataset', 'Imagin…

Textarea(value='SELECT collection_id, COUNT(DISTINCT PatientID) AS patients,\n       COUNT(*) AS series, ROUND…

Button(button_style='primary', description='▶ Run query', style=ButtonStyle())

Output()

**Example · patients per dataset** (edit the query and run it again):

In [16]:
run_sql("""
SELECT collection_id,
       COUNT(DISTINCT PatientID)              AS patients,
       COUNT(*)                               AS series,
       ROUND(SUM(series_size_MB) / 1000, 1)   AS size_GB
FROM index
GROUP BY collection_id
ORDER BY patients DESC
LIMIT 10
""")

,collection_id,patients,series,size_GB
0,nlst,26410,592600,26367.7
1,cbis_ddsm,6671,6775,163.5
2,breast_cancer_screening_dbt,5060,22032,1637.8
3,ccdi_mci,4407,4576,4657.1
4,victre,2994,8749,1027.2
5,eay131,2813,30293,797.3
6,cmmd,1775,1775,22.9
7,covid_19_ny_sbu,1384,17950,511.5
8,prostate_mri_us_biopsy,1151,10373,81.6
9,tcga_brca,1098,10463,3081.9


**Example · join tables: brain MRI at 3 Tesla, with scanner settings.**

In [17]:
run_sql("""
SELECT i.collection_id, i.PatientID, i.SeriesDescription,
       mr.MagneticFieldStrength, mr.MRAcquisitionType,
       ROUND(i.series_size_MB, 1) AS size_MB, i.license_short_name
FROM index i
JOIN mr_index mr USING (SeriesInstanceUID)
JOIN collections_index c USING (collection_id)
WHERE c.tumor_locations ILIKE '%Brain%'
  AND mr.MagneticFieldStrength IN (3, 30000)
  AND i.analysis_result_id IS NULL
""", max_rows=10)

,collection_id,PatientID,SeriesDescription,MagneticFieldStrength,MRAcquisitionType,size_MB,license_short_name
0,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 30ms Flip 30deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
1,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 40ms Flip 30deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
2,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 40ms Flip 90deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
3,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 20ms Flip 30deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
4,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 50ms Flip 60deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
5,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 20ms Flip 60deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
6,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 30ms Flip 60deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
7,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 50ms Flip 30deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
8,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 30ms Flip 90deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0
9,gbm_dsc_mri_dro,GBM_DSC_MRI_DRO_Bo=3T,TR 2sec TE 20ms Flip 90deg -CA pre quarter bol...,3.0,None,8.6,CC BY 4.0


**Only reading is allowed.** Anything else is refused:

In [18]:
run_sql("DELETE FROM index")

---
### What the assistant understands
| You can mention | Examples |
|---|---|
| A dataset id | `rider_pilot`, `nlst`, `tcga_brca` |
| A type of scan | CT, MRI, PET, X-ray, segmentation, pathology slides |
| A body part | chest, lung, brain, head, abdomen, pelvis, breast |
| Sex | male, female, men, women |
| What you want | *how many…*, *which collections…*, *what licence…*, *show me a few…*, *download…* |
| Derived data | *…analysis results* |

If it does not understand, it says so and suggests a question. It is a rule-based
assistant, not a general chatbot, and it only answers from the IDC catalogue.

**More help:** https://learn.canceridc.dev · **Cite IDC:** Fedorov A. et al., RadioGraphics
43(12), 2023, https://doi.org/10.1148/rg.230180, plus each dataset's own DOI.